In [33]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,init_sampler_state,\
    generate_random_initial_states,ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,mcmc_sampler_multichain,\
        NESFermionHopRule,compute_qgt,sampler_info,create_machine_max
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 2  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

In [ ]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz：适配费米子系统的复数值 FFNN"""

    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESTotalAnsatz_stable(nnx.Module):
    def __init__(self, n_spin_orbitals: int, n_states: int = 2, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = n_states
        self.n_spin = n_spin_orbitals

        self.single_ansatz_list = nnx.List()
        key = rngs.params()
        for _ in range(n_states):
            key, sub_key = jax.random.split(key)
            sub_rngs = nnx.Rngs(params=sub_key)
            
            ansatz = SingleStateAnsatz(
                n_spin_orbitals, 
                hidden_dim, 
                rngs=sub_rngs
            )
            self.single_ansatz_list.append(ansatz)
    def __call__(self, x: jax.Array):
        def _forward_single(x_single):
            # 形状：[K, n_spin]
            #print(f'x_single.shape: {x_single.shape}')
            x_single = x_single.reshape(self.K, self.n_spin)
            L = jnp.zeros((self.K, self.K), dtype=complex)
            for i in range(self.K):
                for j in range(self.K):
                    L = L.at[i, j].set(
                        self.single_ansatz_list[j](x_single[i])
                    )
            L_stable = L - L.max()
            sign, log_abs_det = jnp.linalg.slogdet(jnp.exp(L_stable))
            log_Psi_stable = log_abs_det + 1j * jnp.angle(sign)
            return log_Psi_stable, L_stable , L.max()
        
        # 安全的批量处理
        if x.ndim == 2 and x.shape[-1] == self.n_spin:
            # 直接处理单个样本
            return _forward_single(x)
        elif x.ndim == 2 and x.shape[-1] == self.n_spin*self.K:
            x = x.reshape(-1, self.K, self.n_spin)
            # 直接处理批量样本
            return jax.vmap(_forward_single)(x)
        
        elif x.ndim == 3:
            x = x.reshape(-1, self.K, self.n_spin)
            return jax.vmap(_forward_single)(x)
        elif x.ndim ==1:
            x = x[None, :]
            x = x.reshape(self.K, self.n_spin)
            return _forward_single(x)
        else:
            raise ValueError(f'不支持的输入形状: {x.shape}')    

In [ ]:

def create_machine_matrix(model: NESTotalAnsatz):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)

    @jax.jit
    def machine(params, sigma):
        #print(f'x.shape: {sigma.shape}  ')
        m = nnx.merge(graphdef, params)
        log_psi_total,log_M_matrix= m(sigma)
        return log_M_matrix

    return machine, graphdef, state

def create_machine_matrix_max(model: NESTotalAnsatz):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)

    @jax.jit
    def machine(params, sigma):
        #print(f'x.shape: {sigma.shape}  ')
        m = nnx.merge(graphdef, params)
        log_psi_total,log_M_matrix= m(sigma)
        max = log_M_matrix.max(axis=(1,2))
        return max

    return machine, graphdef, state


In [82]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)
total_matirx_max,_,_ = create_machine_max(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=SWEEP_SIZE
)


# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    #'energy_2st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_machine_max=total_max,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    if Natural_Grad == True:
        
        qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.001)
        
        # # 自然梯度求解
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    #history['energy_2st'].append(eig_vals[2])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 5 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        #log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        
        print(f"grad norm = {grad_norm:.4f}")
        print(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha｜1st能量={eig_vals[1]:.8f} Ha｜2st能量={eig_vals[2]:.8f} Ha")
        # print(f'grad={grad_flat[30:31]}')
        print('#-----------------------------------------#')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
print("="*60)


开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha| 第二激发态能量=-0.42938376 Ha
log_Psi: mean=1.172+0.416j | min=-0.165-3.057j | max=1.527+2.763j
grad norm = 0.9061
Step   0 | Loss: -1.2520361220719807|0st能量=-0.99229703 Ha｜1st能量=-0.25973909 Ha｜2st能量=-0.25973909 Ha
#-----------------------------------------#
log_Psi: mean=3.906+0.150j | min=-0.260-1.476j | max=4.240+2.897j
grad norm = 0.5214
Step   5 | Loss: -1.6664048106871645|0st能量=-1.00194422 Ha｜1st能量=-0.66446059 Ha｜2st能量=-0.66446059 Ha
#-----------------------------------------#
log_Psi: mean=2.431+0.186j | min=-0.204-0.240j | max=2.671+2.843j
grad norm = 0.5290
Step  10 | Loss: -1.6740590785910012|0st能量=-0.95270149 Ha｜1st能量=-0.72135759 Ha｜2st能量=-0.72135759 Ha
#-----------------------------------------#
log_Psi: mean=0.627-0.115j | min=-0.473-2.307j | max=1.288+2.741j
grad norm = 2.4292
Step  15 | Loss: -1.6778060503734562|0st能量=-0.90752352 Ha｜1st能量=-0.77028253 Ha｜2st能量=-0.77028253 Ha
#-----------------

In [ ]:
Natural_Grad = True

$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [ ]:
import pickle
import os  # 加上这个
# 自动创建 data 文件夹（关键修复）
os.makedirs('./data', exist_ok=True)
if Natural_Grad == True:
    print('保存自然梯度历史记录')
    # 保存 history
    with open('./data/history_natural_gradient_K2.pkl', 'wb') as f:
        pickle.dump(history, f)
else:
    # 保存 history
    with open('./data/history_plain_gradient_K2.pkl', 'wb') as f:
        pickle.dump(history, f)

print("保存成功！")

In [ ]:
import pickle
history_natural= pickle.load(open('./data/history_natural_gradient_K2.pkl', 'rb'))
history_plain= pickle.load(open('./data/history_plain_gradient_K2.pkl', 'rb'))


In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(3, 2, figsize=(12, 9))
fig.suptitle('Natural Excited State-VMC for $H_2$ K=2 ')
# 第一个子图
axs[0,0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0,0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0,0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0,0].set_title('0st Energy')
axs[0,0].set_ylabel('energy')
axs[0,0].set_xlabel('step')
axs[0,0].set_xlabel('step')
axs[0,0].legend()

axs[0,1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[0,1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[0,1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[0,1].set_title('1st Energy')
axs[0,1].set_ylabel('energy')
axs[0,1].set_xlabel('step')      
axs[0,1].legend()

# # 第二个子图
axs[1,0].plot(history_natural['energy_0st']-E_fcis[0],color='orange',label='natural gradient')    
axs[1,0].set_title('0st Energy Error')
axs[1,0].set_xlabel('step')
axs[1,0].set_ylabel('energy')
axs[1,0].legend()



axs[1,1].plot(history_natural['energy_1st']-E_fcis[1],color='orange',label='natural gradient')    
axs[1,1].set_title('1st Energy Error')
axs[1,1].set_xlabel('step')
axs[1,1].set_ylabel('energy')
axs[1,1].legend()

# # 第二个子图
axs[2,0].plot(history_natural['loss'],color='orange',label='natural gradient')  
axs[2,0].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[2,0].set_title('loss')
axs[2,0].set_xlabel('step')
axs[2,0].set_ylabel('loss')
axs[2,0].legend()

axs[2,1].plot(history_natural['grad_norm'][15:],color='orange',label='natural gradient')  
axs[2,1].plot(history_plain['grad_norm'][15:],color='blue',label='plain gradient')
axs[2,1].set_title('grad_norm')
axs[2,1].set_xlabel('step')
axs[2,1].set_ylabel('grad_norm')
axs[2,1].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('NES-VMC for $H_2$ K=3 ')
# 第一个子图
axs[0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')
axs[0].legend()

axs[1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')      
axs[1].legend()

# axs[2].plot(history_natural['energy_2st'],color='orange',label='natural gradient')
# axs[2].plot(history_plain['energy_2st'],color='blue',label='plain gradient')
# axs[2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red')
# axs[2].set_title('2st Energy')
# axs[2].set_ylabel('energy')
# axs[2].set_xlabel('step')      
# axs[2].legend()


# 第二个子图
axs[2].plot(history_natural['loss'],color='orange',label='natural gradient')            
axs[2].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[2].set_title('loss')
axs[2].set_xlabel('step')
axs[2].set_ylabel('loss')
axs[2].legend()

# 第三个子图
axs[3].plot(history_natural['grad_norm'],color='orange',label='natural gradient')
axs[3].plot(history_plain['grad_norm'],color='blue',label='plain gradient')
axs[3].set_title('grad_norm')
axs[3].set_xlabel('step')
axs[3].set_ylabel('grad_norm')
axs[3].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt

history = history_natural
# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle('NES-VMC for $H_2$ K=2 Natural Gradient Descent')
# 第一个子图
axs[0].plot(history['energy_0st'])
axs[0].hlines(E_fcis[0],0,len(history['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')

axs[1].plot(history['energy_1st'])
axs[1].hlines(E_fcis[1],0,len(history['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')

# 第二个子图
axs[2].plot(history['loss'])
axs[2].set_title('loss')
axs[2].set_xlabel('step')
axs[2].set_ylabel('energy')

# 第三个子图
axs[3].plot(history['grad_norm'])
axs[3].set_title('grad_norm')
axs[3].set_xlabel('step')
axs[3].set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt

#history = history_natural
# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle('NES-VMC for $H_2$ K=2 Natural Gradient Descent')
# 第一个子图
axs[0].plot(history['energy_0st'])
axs[0].hlines(E_fcis[0],0,len(history['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')

axs[1].plot(history['energy_1st'])
axs[1].hlines(E_fcis[1],0,len(history['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')

# 第二个子图
axs[2].plot(history['loss'])
axs[2].set_title('loss')
axs[2].set_xlabel('step')
axs[2].set_ylabel('energy')

# 第三个子图
axs[3].plot(history['grad_norm'])
axs[3].set_title('grad_norm')
axs[3].set_xlabel('step')
axs[3].set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

剖析为啥有毛病 

In [ ]:
history['energy_0st'][40:45]

In [ ]:
from collections import Counter
import numpy as np
def sampler_info(samples:jnp.array,K:int):
    test_samples = np.array(samples.reshape(-1, 4*K))
    count = Counter(tuple(each_row.tolist()) for each_row in test_samples)
    for tpl, count_ in count.items():
        print(f"元组 {tpl} 出现了 {count_} 次")
    return count


sampler_info(history['samples'][100],K)

In [ ]:
sampler_info(history['samples'][102],K)

In [ ]:

print(f'我当时保存的: loss: {history["loss"][102]:.3f}|energy_0st: {history["energy_0st"][102]:.3f}|grad_norm: {history["grad_norm"][102]:.3f}')
test_samples = history['samples'][102]
test_params = history['params'][101]
# 3. 计算能量和自然梯度（逻辑和原代码一致）
grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                total_matrix_machine=total_matrix_machine,
                                                total_machine=total_machine,
                                                single_machine_list=single_machine_list,
                                                total_params=test_params,
                                                x_batch=test_samples.reshape(-1,K,4))

grad_flat , grad_unravel_fn = ravel_pytree(grad)
grad_norm = jnp.linalg.norm(grad_flat)

eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
print(f'我基于当时的 Samples 尝试复现:')
print(f"grad_norm: {grad_norm:.3f}|loss_mean: {loss_mean:.3f}|energy_0st: {eig_vals[0]:.3f}")
print(E_L_mean)

In [ ]:
from NES_VMC import NES_loss_energy

trace, E_L = NES_loss_energy(ha=ha,
                total_matrix_machine=total_matrix_machine,
                single_machine_list=single_machine_list,
                total_params=total_params,
                x=test_samples.reshape(-1,2,4))

trace

In [ ]:
test_samples.reshape(-1,2,4)[0]

In [ ]:
E_L[0]

In [ ]:
def NES_loss_energy(ha, total_matrix_machine,single_machine_list,total_params, x):
    log_M = total_matrix_machine(total_params,x)
    Psi_Matrix = jnp.exp(log_M)
    # 添加正则化项，防止矩阵奇异
    #Psi_Matrix += 1e-6 * jnp.eye(Psi_Matrix.shape[0])
    H_psi_x = Ham_Psi(ha,single_machine_list,total_params,x)
    Psi_Matrix_inv = jnp.linalg.solve(Psi_Matrix, H_psi_x)
    return jnp.real(jnp.trace(Psi_Matrix_inv, axis1=-2, axis2=-1)), Psi_Matrix_inv

In [ ]:
total_matrix_machine(total_params, history['samples'][41].reshape(-1,K,4))

In [ ]:
NES_loss_energy(ha=ha,total_matrix_machine=total_matrix_machine,
                single_machine_list=single_machine_list,
                total_params=total_params,
                x=abnormal_samples.reshape(-1,K,4))


In [ ]:
M = jnp.array([[2,2],[3,4]])
jnp.log(jnp.linalg.det(M)) #Array(0.69314718, dtype=float64)
jnp.trace(jnp.log(M)) #Array(2.07944154, dtype=float64)

In [ ]:
jnp.trace(jnp.log(M))
